<a href="https://colab.research.google.com/github/robertkigobe/llm_engineering/blob/main/cds_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hardware specifications

In [1]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

Wed Mar 18 20:20:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             17W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup & installs

In [2]:
!pip -q install \
  pandas gradio transformers accelerate sentencepiece \
  langchain langchain-core langchain-huggingface

# Imports & global

In [3]:
import pandas as pd
import gradio as gr
from datetime import datetime

from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline


# Load your log data

In [4]:
import pandas as pd

LOG_DF = None  # global dataset used by the rest of the demo

def load_logs_gradio(file):
    global LOG_DF

    if file is None:
        return pd.DataFrame({"message": ["No file uploaded yet."]})

    path = file.name  # temp path in Colab runtime

    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl") or path.endswith(".ndjson"):
        df = pd.read_json(path, lines=True)
    elif path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        return pd.DataFrame({"error": [f"Unsupported file type: {path}"]})

    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)

    LOG_DF = df
    return df.head(15)

def dataset_profile():
    if LOG_DF is None:
        return "No dataset loaded."
    return (
        f"Loaded ✅ Rows: {len(LOG_DF):,} | Columns: {len(LOG_DF.columns)}\n\n"
        "Columns:\n- " + "\n- ".join(map(str, LOG_DF.columns))
    )


In [5]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Log File Loader (Analyst Demo)")

    with gr.Row():
        with gr.Column(scale=1):
            file_in = gr.File(label="Choose a log file (.csv, .jsonl, .xlsx)")
            profile_btn = gr.Button("Show dataset info")

        with gr.Column(scale=2):
            preview = gr.Dataframe(label="Preview (first rows)")
            info = gr.Markdown()

    file_in.upload(load_logs_gradio, inputs=file_in, outputs=preview)
    profile_btn.click(dataset_profile, outputs=info)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0c84f445a29789f4b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Normalize & enrich logs (truth layer)

In [6]:
def normalize_logs(df):
    df = df.copy()
    df["date"] = df["@timestamp"].dt.date
    df["hour"] = df["@timestamp"].dt.floor("H")
    df["statusClass"] = (df["statusCode"] // 100).astype(str) + "xx"
    return df

# Analyst‑safe query functions

In [7]:
# ================================
# COLAB CELL 5
# Analyst-safe query functions
# ================================

import pandas as pd

# ---- Safety check ----
def ensure_data_loaded():
    if uploaded_df is None or uploaded_df.empty:
        return False, "❌ No data loaded yet. Please upload a log file first."
    return True, None


# ---- Helper: date parsing ----
def normalize_timestamp(df):
    if not pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    return df


# ======================================================
# SAFE QUERY 1: High-level error summary
# ======================================================
def get_error_summary():
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df.copy())

    summary = (
        df[df["log_level"] == "ERROR"]
        .groupby("error_type")
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )

    return summary


# ======================================================
# SAFE QUERY 2: Errors by service
# ======================================================
def get_errors_by_service(service_name):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df.copy())

    filtered = df[
        (df["log_level"] == "ERROR") &
        (df["service"] == service_name)
    ]

    if filtered.empty:
        return f"✅ No errors found for service: {service_name}"

    return filtered[[
        "timestamp",
        "service",
        "error_type",
        "message",
        "sourceId"
    ]].sort_values("timestamp", ascending=False).head(50)


# ======================================================
# SAFE QUERY 3: Errors in a time range
# ======================================================
def get_errors_in_date_range(start_date, end_date):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df.copy())

    mask = (
        (df["timestamp"] >= pd.to_datetime(start_date)) &
        (df["timestamp"] <= pd.to_datetime(end_date)) &
        (df["log_level"] == "ERROR")
    )

    result = df.loc[mask]

    if result.empty:
        return "✅ No errors found in the selected date range."

    return result[[
        "timestamp",
        "service",
        "error_type",
        "message"
    ]].sort_values("timestamp", ascending=False).head(100)


# ======================================================
# SAFE QUERY 4: Most common errors (executive-friendly)
# ======================================================
def get_top_errors(limit=5):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = uploaded_df.copy()

    top_errors = (
        df[df["log_level"] == "ERROR"]
        .groupby(["error_type", "service"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(limit)
    )

    return top_errors


# ======================================================
# SAFE QUERY 5: Plain-English explanation helper
# ======================================================
def explain_error_type(error_type):
    explanations = {
        "TimeoutException": "The service took too long to respond.",
        "NullPointerException": "The system tried to use missing data.",
        "DatabaseConnectionError": "The service could not reach the database.",
        "AuthFailure": "Authentication or authorization failed.",
        "ServiceUnavailable": "The service was temporarily down."
    }

    return explanations.get(
        error_type,
        "No predefined explanation available for this error type."
    )


# Load Hugging Face model (small & demo‑friendly)

In [8]:
# ================================
# COLAB CELL 6 — FLAN-T5 (version-proof, no HF pipeline registry)
# ================================

!pip -q install -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)  # <-- do NOT flip tie_word_embeddings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def llm_invoke(prompt: str, max_new_tokens: int = 256) -> str:
    """Small, reliable 'invoke' for FLAN-T5 without pipelines or LangChain."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic for demo
            num_beams=2,              # a little better quality than greedy, still fast
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Smoke test
print(llm_invoke("Summarize: Service A has TimeoutException spikes after a deployment."))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Service A has a timeoutException spike after a deployment.


# LLM prompt for analyst explanations

In [9]:
# ================================# COLAB CELL 7 — Analyst‑Safe Log Explorer (Gradio)
# ================================

!pip -q install -U gradio

import gradio as gr
import pandas as pd

# ------------------------------------------------
# Ensure uploaded_df always exists
# ------------------------------------------------
try:
    uploaded_df
except NameError:
    uploaded_df = None


# ------------------------------------------------
# Helpers
# ------------------------------------------------
def _ensure_loaded():
    return isinstance(uploaded_df, pd.DataFrame) and not uploaded_df.empty


def _services():
    if not _ensure_loaded() or "service" not in uploaded_df.columns:
        return []
    return sorted(uploaded_df["service"].dropna().unique().tolist())


def _error_types():
    if not _ensure_loaded() or "error_type" not in uploaded_df.columns:
        return []
    return sorted(uploaded_df["error_type"].dropna().unique().tolist())


def _df_and_markdown(result, max_rows=15):
    if isinstance(result, str):
        return pd.DataFrame(), result

    if result is None or result.empty:
        return pd.DataFrame(), "✅ No results."

    md = result.head(max_rows).to_markdown(index=False)
    return result, md


# ------------------------------------------------
# Safe query handlers
# ------------------------------------------------
def ui_error_summary():
    return _df_and_markdown(get_error_summary())


def ui_top_errors(limit):
    return _df_and_markdown(get_top_errors(int(limit)))


def ui_errors_by_service(service):
    if not service:
        return pd.DataFrame(), "Pick a service first."
    return _df_and_markdown(get_errors_by_service(service))


def ui_errors_in_range(start_date, end_date):
    if not start_date or not end_date:
        return pd.DataFrame(), "Pick both start and end dates."
    return _df_and_markdown(get_errors_in_date_range(start_date, end_date))


# ------------------------------------------------
# LLM (explain only — no querying)
# ------------------------------------------------
def ui_explain_error_type(error_type):
    if not error_type:
        return "Pick an error type first."

    base = ""
    try:
        base = explain_error_type(error_type)
    except Exception:
        pass

    prompt = (
        "Explain this software error type in plain English for a non‑technical analyst.\n"
        "Limit to 2–3 sentences and suggest one next check.\n\n"
        f"Error type: {error_type}\n"
        f"Known hint: {base}"
    )

    try:
        return llm_invoke(prompt, max_new_tokens=120)
    except Exception as e:
        return f"{base}\n\n(LLM unavailable: {e})"


def ui_narrate_table(table):
    if not isinstance(table, pd.DataFrame) or table.empty:
        return "Run a query first."

    csv_text = table.head(25).to_csv(index=False)
    prompt = (
        "Summarize the table below for a non‑technical analyst.\n"
        "Call out the biggest pattern and one recommended next step.\n\n"
        f"{csv_text}"
    )

    try:
        return llm_invoke(prompt, max_new_tokens=200)
    except Exception as e:
        return f"(LLM unavailable: {e})"


# ------------------------------------------------
# Dropdown refresh (THIS is the key fix)
# ------------------------------------------------
def refresh_dropdowns():
    return (
        gr.update(choices=_services()),
        gr.update(choices=_error_types())
    )


# ------------------------------------------------
# Build UI
# ------------------------------------------------
with gr.Blocks(title="Analyst‑Safe Log Explorer") as demo:
    gr.Markdown(
        "# Analyst‑Safe Log Explorer\n"
        "Guided log analysis for analysts. "
        "**No raw querying, no Python, no SQL.**"
    )

    # ----------------------------
    # Safe Queries
    # ----------------------------
    with gr.Tab("Safe Queries"):
        with gr.Row():
            btn_summary = gr.Button("Error Summary", variant="primary")
            btn_top = gr.Button("Top Errors")
            top_k = gr.Slider(3, 20, value=5, step=1, label="Top K")

        with gr.Row():
            service_dd = gr.Dropdown(
                label="Service",
                choices=[],
                allow_custom_value=True
            )
            refresh_btn = gr.Button("🔄 Refresh Services")
            btn_service = gr.Button("Errors by Service")

        with gr.Row():
            start = gr.Textbox(label="Start date (YYYY‑MM‑DD)")
            end = gr.Textbox(label="End date (YYYY‑MM‑DD)")
            btn_range = gr.Button("Errors in Date Range")

        result_table = gr.Dataframe(label="Results", interactive=False)
        result_md = gr.Markdown()

        btn_summary.click(ui_error_summary, outputs=[result_table, result_md])
        btn_top.click(ui_top_errors, inputs=top_k, outputs=[result_table, result_md])
        btn_service.click(ui_errors_by_service, inputs=service_dd, outputs=[result_table, result_md])
        btn_range.click(ui_errors_in_range, inputs=[start, end], outputs=[result_table, result_md])

        refresh_btn.click(refresh_dropdowns, outputs=[service_dd, service_dd])

    # ----------------------------
    # Explain (LLM)
    # ----------------------------
    with gr.Tab("Explain (LLM)"):
        gr.Markdown("### Plain‑English explanations (LLM does not query data)")

        err_dd = gr.Dropdown(
            label="Error Type",
            choices=[],
            allow_custom_value=True
        )
        refresh_err_btn = gr.Button("🔄 Refresh Error Types")

        explain_btn = gr.Button("Explain Error Type", variant="primary")
        explain_out = gr.Markdown()

        explain_btn.click(ui_explain_error_type, inputs=err_dd, outputs=explain_out)
        refresh_err_btn.click(refresh_dropdowns, outputs=[service_dd, err_dd])

        gr.Markdown("### Narrate last results table")
        narrate_btn = gr.Button("Narrate Results")
        narrate_out = gr.Markdown()

        narrate_btn.click(ui_narrate_table, inputs=result_table, outputs=narrate_out)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ceb1212b15069310a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
